In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
# =========================
# Load metadata
# =========================

df = pd.read_csv("../processed/all_data/metadata.csv")

In [3]:
# =========================
# Create patient-level dataframe
# =========================

patient_df = df.groupby("patient_id").agg({
    "classification": "first"
}).reset_index()

In [4]:
# =========================
# Train / Temp split
# 80% train (with forced IDs)
# 20% temp
# =========================

# Force specific patient IDs into training split
force_train_ids = {"PN5617"}

forced_train_patients = patient_df[patient_df["patient_id"].isin(force_train_ids)]
remaining_patients = patient_df[~patient_df["patient_id"].isin(force_train_ids)]

train_patients, temp_patients = train_test_split(
    remaining_patients,
    test_size=0.2,
    stratify=remaining_patients["classification"],
    random_state=42
)

# Add forced patients back into train
train_patients = pd.concat([train_patients, forced_train_patients], ignore_index=True)

In [5]:
# =========================
# Validation / Test split
# 10% val
# 10% test
# =========================

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.5,
    stratify=temp_patients["classification"],
    random_state=42
)


In [6]:
# =========================
# Get patient ID sets
# =========================

train_ids = set(train_patients["patient_id"])
val_ids = set(val_patients["patient_id"])
test_ids = set(test_patients["patient_id"])

In [7]:
# =========================
# Split original dataframe
# =========================

train_df = df[df["patient_id"].isin(train_ids)]
val_df = df[df["patient_id"].isin(val_ids)]
test_df = df[df["patient_id"].isin(test_ids)]

In [8]:
# =========================
# Save splits
# =========================

train_df.to_csv("../processed/all_data/train.csv", index=False)
val_df.to_csv("../processed/all_data/val.csv", index=False)
test_df.to_csv("../processed/all_data/test.csv", index=False)

In [9]:
# =========================
# Print statistics
# =========================

print("\n===== TRAIN =====")
print(train_df["classification"].value_counts())
print("Total: " + str(len(train_df)))

print("\n===== VALIDATION =====")
print(val_df["classification"].value_counts())
print("Total: " + str(len(val_df)))

print("\n===== TEST =====")
print(test_df["classification"].value_counts())
print("Total: " + str(len(test_df)))

print("\nSplits saved successfully.")


===== TRAIN =====
classification
Normal          12460
Pneumonia        4494
COVID-19         2893
Tuberculosis      969
Name: count, dtype: int64
Total: 20816

===== VALIDATION =====
classification
Normal          1558
Pneumonia        562
COVID-19         361
Tuberculosis     121
Name: count, dtype: int64
Total: 2602

===== TEST =====
classification
Normal          1557
Pneumonia        562
COVID-19         362
Tuberculosis     121
Name: count, dtype: int64
Total: 2602

Splits saved successfully.


In [10]:
# Check no patient overlap
assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

# Check forced patient IDs are in train
assert force_train_ids.issubset(train_ids)
assert force_train_ids.isdisjoint(val_ids)
assert force_train_ids.isdisjoint(test_ids)

print("No patient leakage detected.")
print("Forced patient IDs are in train split.")

No patient leakage detected.
Forced patient IDs are in train split.
